(z3-gray-code)=
# 格雷码：让相邻两个数只差一个二进制位的排列方法

## 这一章要搭的数列

格雷码这种数列，排法很有讲究，相邻的两个值只差一个二进制位。纠错、编码、电路设计、模拟与数字之间的转换，这些地方都要用到它。

- 4 位的话，就是把 0～15 每个数排一次，让相邻两个之间只差 1 位
- 先用 `Int`(整数)写一遍，再换成 `BitVec` 提速
- 同一道题，仅仅把变量的类型换掉，就可能快出 100 倍以上，下面会看到这一点

[用 OrTools 求解格雷码](ortools-gray-code)

## 先想清楚要满足什么

$n$ 位的格雷码，把 $0$ 到 $2^n-1$ 这么多数一个不落地排一遍，还得同时满足下面两条。

- 相邻的两个数，恰好只差 1 位
- 末尾和开头也恰好只差 1 位(绕一圈还能接回去)

用 `Int` 来写，就得动手把 `x` 的第 `i` 位取出来，而 `x % 2**(i+1) - x % 2**i` 这个式子，第 `i` 位是 `0` 就给 `0`，是 `1` 就给 `2**i`，因此两个数这么一算结果不同，就说明这一位不同。

## 先用 Int 类型的变量求解

In [1]:
from z3 import *


def gray_code(nbits):
    """返回 nbits 位的格雷码（数的排列）"""
    n = 2 ** nbits
    codes = IntVector('code', n)
    solver = Solver()

    for c in codes:
        solver.add(0 <= c, c < n)
    solver.add(Distinct(codes))

    for c1, c2 in zip(codes, codes[1:] + codes[:1]):
        diff = Abs(c1 - c2)
        # 辅助条件：只差 1 位时，差值是 2 的幂
        solver.add(Or([diff == 2 ** bit for bit in range(nbits)]))
        # 真正的条件：不同的位恰好 1 个
        bit_diff = [c1 % 2**(i + 1) - c1 % 2**i != c2 % 2**(i + 1) - c2 % 2**i
                    for i in range(nbits)]
        solver.add(Sum(bit_diff) == 1)

    solver.add(codes[0] == 0)      # 为了消除旋转，把开头固定为 0

    assert solver.check() == sat
    model = solver.model()
    return [model[c].as_long() for c in codes]


nbits = 4
%time codes = gray_code(nbits)
for c in codes:
    print(f'{c:0{nbits}b}')

CPU times: total: 9.19 s
Wall time: 9.33 s
0000
0001
1001
1011
0011
0111
1111
1110
1010
1000
1100
1101
0101
0100
0110
0010


0 到 15 一个不落，相邻之间只差 1 位，这样的数列就出来了。4 位正好对应 16 个数，结果与预期完全一致。

可只要用 `Int` 写，Z3 就跑得慢，在这个环境里它要花几秒才能给出结果。原因不难推测：由于位的操作被改写成了数的除法，式子变得别扭，Z3 处理起来相当费劲。

## 换成 BitVec 类型提高速度

既然是跟位打交道的题，一开始就应当用位向量类型 `BitVec`，这才是这一类问题应当选用的类型。`Int` 一旦换成 `BitVec`，下面两件事都变得简单。

- 想取某一位，`>>` 配 `&` 即可
- 两个数哪里不同，`^`(XOR)一下就出来

`c1 ^ c2` 得到的数，只有那些不同的位是 1。放到格雷码上，它就必然是 2 的幂，即一串 0 里只有孤零零的一个 1。

```python
diff = c1 ^ c2
solver.add(diff & (diff - 1) == 0, diff != 0)
```

`diff & (diff - 1) == 0` 是判断“`diff` 是不是 2 的幂”的常用方法。2 的幂减掉 1，立着的那一位退成 0，它下面的位全翻成 1，再跟原数做 AND，结果自然是 0。后面那个 `diff != 0`，是把“0 位不同(也就是同一个数)”挡在外面。

In [2]:
def gray_code_bv(nbits):
    """BitVec 版：用位运算解同一个问题"""
    codes = [BitVec(f'code_{i}', nbits) for i in range(2 ** nbits)]
    solver = Solver()
    solver.add(Distinct(codes))

    for c1, c2 in zip(codes, codes[1:] + codes[:1]):
        diff = c1 ^ c2                       # 只有不同的位为 1
        solver.add(diff & (diff - 1) == 0,   # 2 的幂
                   diff != 0)                # 不是 0

    solver.add(codes[0] == 0)

    assert solver.check() == sat
    model = solver.model()
    return [model[c].as_long() for c in codes]


%time codes = gray_code_bv(nbits)
for c in codes:
    print(f'{c:0{nbits}b}')

CPU times: total: 15.6 ms
Wall time: 22.5 ms
0000
1000
1100
0100
0101
0111
0110
1110
1111
1101
1001
1011
1010
0010
0011
0001


这一回的 `BitVec` 版一瞬间就结束了，在这个环境里 `Int` 版要 10 秒上下，`BitVec` 版大约 0.03 秒，差了 300 倍以上(`Int` 版每次执行，几秒到十几秒不等，但量级不会变)。

两次的模型完全相同，差别只在于变量的类型。这一次排出来的顺序，与 `Int` 版未必相同，可两边都是正确的格雷码。满足条件的排法本来就不止一种，因此两边的顺序不同并不奇怪。

## 小结

- 格雷码说的是“相邻两个只差 1 位”的那种排法
- 用 `Int` 写，得凑成“差值是 2 的幂”加上“不同的位正好 1 个”
- 跟位有关的题，换成 `BitVec` 就快，XOR 一个式子就把差异说清了
- `diff & (diff - 1) == 0` 是判断“是不是 2 的幂”的技巧
- 答案不止一个，Z3 返回的是它最先遇到的那一个